# Cahn-Hilliard PINN — PyTorch Implementation

**PDE (Equation 6, PhysRevE.106.065303):**

$$\frac{\partial U}{\partial t} = D\,\nabla^2\!\bigl(\nabla^2 U + a_2\,U + a_4\,U^3\bigr)$$

**Approach:** Physics-Informed Neural Network trained in two phases:
1. **Adam** warmup (first-order, fast)
2. **Self-Scaled Broyden** quasi-Newton (second-order, precise)

All hyperparameters, PDE coefficients, domain bounds, and training options
are loaded from a **YAML configuration file** so that experiments can be
launched by simply swapping configs — no code edits required.

In [ ]:
# ============================================================================
# CELL 1: IMPORTS AND ENVIRONMENT SETUP
# ============================================================================
# This cell imports every library the notebook uses.  Nothing is trained or
# computed here — it just makes functions and classes available.
# ============================================================================

# --- Standard library -------------------------------------------------------
import os                # Operating-system utilities: file paths, makedirs
import math              # Pure-Python math: sqrt, constants (used in init)
import yaml              # YAML parser: reads the experiment config file
from time import perf_counter  # High-resolution wall-clock timer for profiling

# --- Numerical / scientific computing --------------------------------------
import numpy as np       # N-dimensional arrays, random sampling, linear algebra
import scipy.io          # (unused here but kept for .mat file I/O if needed)
from scipy.optimize import minimize          # General-purpose optimizer (BFGS, L-BFGS-B)
from scipy.linalg import cholesky, LinAlgError  # Cholesky decomposition for PD check
from scipy.interpolate import RegularGridInterpolator  # Smooth grid interpolation (IC)
from scipy.ndimage import gaussian_filter    # Gaussian smoothing of the IC field

# --- Deep learning (PyTorch) -----------------------------------------------
import torch             # Tensor operations, autograd, GPU acceleration
import torch.nn as nn    # Neural-network building blocks (Linear, Sequential, …)
from torch.nn.utils import parameters_to_vector, vector_to_parameters
#   parameters_to_vector : flattens all net params into a single 1-D tensor
#   vector_to_parameters : loads a flat vector back into the net's param buffers
#   → used to interface PyTorch ↔ SciPy (which works with flat numpy arrays)

# --- Visualisation ----------------------------------------------------------
import matplotlib.pyplot as plt   # Plotting API (loss curves, snapshots, etc.)
from matplotlib import cm         # Colour maps (not used directly but available)

# ---- Path to the YAML configuration file -----------------------------------
# This single variable controls the ENTIRE experiment.  Change it to point at
# a different YAML file and the notebook will use those hyperparameters,
# PDE coefficients, domain bounds, network shape, etc.  No other code
# edits are needed.
CONFIG_PATH = "configs/cahn_hilliard_canonical.yaml"

In [ ]:
# ============================================================================
# CELL 2: LOAD CONFIGURATION AND SET ALL PARAMETERS
# ============================================================================
# Opens the YAML file specified by CONFIG_PATH and unpacks every setting
# into plain Python variables.  This is the ONLY place that reads the
# config — all later cells just use these variables.
#
# Why YAML?  It lets us define a new experiment (different optimizer,
# learning rate, network size, …) by writing a short text file instead
# of editing code.  The 17 configs in configs/ use this mechanism.
# ============================================================================

# --- Read the YAML file into a nested Python dict --------------------------
with open(CONFIG_PATH, "r") as f:
    cfg = yaml.safe_load(f)   # cfg is a dict-of-dicts matching the YAML structure

# --- Equation coefficients --------------------------------------------------
# Cahn-Hilliard:  ∂U/∂t = D · ∇²( ∇²U + a₂·U + a₄·U³ )
D  = cfg["equation"]["D"]     # Diffusion / mobility constant  (canonical: 1.0)
a2 = cfg["equation"]["a2"]    # Quadratic coeff in chemical potential (canon: 1.0)
a4 = cfg["equation"]["a4"]    # Quartic  coeff in chemical potential (canon: 1.0)

# --- Spatial / temporal domain ----------------------------------------------
# The PDE is solved on the rectangle [x_min, x_max] × [y_min, y_max]
# over the time interval [t_min, t_max].
x_min = cfg["domain"]["x_min"]   # Left   boundary  (canonical: 0)
x_max = cfg["domain"]["x_max"]   # Right  boundary  (canonical: 1)
y_min = cfg["domain"]["y_min"]   # Bottom boundary  (canonical: 0)
y_max = cfg["domain"]["y_max"]   # Top    boundary  (canonical: 1)
t_min = cfg["domain"]["t_min"]   # Start time       (canonical: 0)
t_max = cfg["domain"]["t_max"]   # End time         (canonical: 20)
Lx = x_max - x_min              # Domain length in x (used by samplers)
Ly = y_max - y_min              # Domain length in y
normalize_inputs = cfg["domain"].get("normalize_inputs", True)
#   If True, an InputNormalization layer maps (t, x, y) → [-1, 1]³
#   before the first hidden layer.  Helps gradient flow in deep nets.

# --- Initial condition ------------------------------------------------------
# We generate a random field U₀(x, y) on a grid and smooth it with a
# Gaussian filter.  The PINN must learn to match this IC at t = t_min.
ic_cfg        = cfg["initial_condition"]       # Sub-dict with IC settings
ic_type       = ic_cfg["type"]                 # "random_smooth" (only type so far)
ic_seed       = ic_cfg.get("seed", 42)         # RNG seed → reproducible IC
ic_grid_nx    = ic_cfg.get("grid_nx", 128)     # Grid resolution in x
ic_grid_ny    = ic_cfg.get("grid_ny", 128)     # Grid resolution in y
ic_smooth_sig = ic_cfg.get("smoothing_sigma", 2.0)
#   Gaussian kernel width (in grid cells).  Larger = smoother IC.

# --- Boundary conditions ---------------------------------------------------
bc_cfg         = cfg["boundary_conditions"]
bc_type        = bc_cfg["type"]                # "periodic" — the only type used
bc_enforcement = bc_cfg.get("enforcement", "soft")
#   "soft" = BC mismatch is penalised in the loss (the standard PINN way)
bc_deriv_order = bc_cfg.get("derivative_order", 0)
#   0 → match values only:  U(left) = U(right)
#   1 → also match 1st normal derivatives: U_x(left) = U_x(right), etc.
#   Higher orders would need extra autograd in compute_bc_derivatives.

# --- Network architecture --------------------------------------------------
net_cfg         = cfg["network"]
layer_dims      = tuple(net_cfg["layer_dims"])
#   e.g. (3, 20, 20, 20, 20, 1):  3 inputs → 4 hidden layers of 20 → 1 output
activation_name = net_cfg.get("activation", "tanh")
#   Activation after every hidden layer.  tanh is smooth → good for 4th-order PDE.
output_scaling  = net_cfg.get("output_scaling", 1.0)
#   Multiplies the final output.  Can help if the solution has a different scale.

# --- Training schedule: Adam (phase 1) ------------------------------------
adam_cfg       = cfg["training"]["adam"]
Nepochs_ADAM   = adam_cfg["epochs"]             # Number of Adam iterations
adam_lr        = adam_cfg["lr"]                  # Initial learning rate
adam_betas     = tuple(adam_cfg["betas"])        # (β₁, β₂) momentum parameters
adam_eps       = adam_cfg["eps"]                 # Numerical stability in Adam
lr_decay_rate  = adam_cfg.get("lr_decay_rate", 0.98)
#   Multiplicative factor per lr_decay_steps epochs (exponential decay)
lr_decay_steps = adam_cfg.get("lr_decay_steps", 1000)
#   How many epochs correspond to one decay factor application

# --- Training schedule: BFGS / L-BFGS-B (phase 2) -------------------------
bfgs_cfg       = cfg["training"]["bfgs"]
Nbfgs          = bfgs_cfg["epochs"]             # Total quasi-Newton iterations
Nchange        = bfgs_cfg["batch_size"]
#   After every Nchange BFGS iterations, resample collocation points (RAD).
bfgs_method    = bfgs_cfg["method"]             # "BFGS" or "L-BFGS-B"
bfgs_variant   = bfgs_cfg.get("variant", "none")
#   For dense BFGS: SSBroyden2, SSBFGS_OL, SSBFGS_AB, SSBroyden1, BFGS_scipy
#   Ignored when method is L-BFGS-B.
bfgs_init_scale = bfgs_cfg.get("initial_scale", False)
#   Whether to scale the initial Hessian approximation (first batch only).
bfgs_power     = bfgs_cfg.get("power", 1.0)
#   Power transform: minimise L^(1/p) instead of L.  p > 1 flattens the
#   loss landscape, which can improve conditioning for quasi-Newton.
bfgs_maxcor    = bfgs_cfg.get("maxcor", 20)
#   L-BFGS-B only: number of (s, y) correction pairs to store.

# --- Sampling ---------------------------------------------------------------
samp_cfg   = cfg["sampling"]
Nint       = samp_cfg["n_interior"]       # Interior collocation points (PDE residual)
N0         = samp_cfg["n_initial"]        # Initial-condition points
Nb         = samp_cfg["n_boundary"]       # Boundary points (periodic BC)
Nresample  = samp_cfg["resample_every"]   # Re-draw points every this many epochs/iters

# RAD = Residual-Adaptive Distribution: focus points where residual is large
rad_cfg    = samp_cfg.get("rad", {})
rad_on     = rad_cfg.get("enabled", True)  # Turn RAD on/off
k1         = rad_cfg.get("k1", 1.0)        # Exponent on |residual|: w = |r|^k1
k2         = rad_cfg.get("k2", 1.0)        # Additive offset in probability: p ∝ w/mean(w) + k2
Nsampling  = rad_cfg.get("n_candidates", 50000)
#   Number of candidate points to evaluate before subsampling by weight.
rad_args   = (k1, k2)                      # Packed for passing to adaptive_rad()

# --- Loss weights -----------------------------------------------------------
lw         = cfg["loss_weights"]
lam_pde    = lw.get("pde", 1.0)    # Weight on L_PDE (PDE residual MSE)
lam_ic     = lw.get("ic", 5.0)     # Weight on L_IC  (initial condition MSE)
lam_bc     = lw.get("bc", 5.0)     # Weight on L_BC  (periodic boundary MSE)
#   IC and BC weights > PDE weight because matching data is easier but critical.

# --- Logging ----------------------------------------------------------------
log_cfg     = cfg.get("logging", {})
Nprint      = log_cfg.get("print_every", 100)   # Print loss every Nprint iterations
RESULTS_DIR = log_cfg.get("results_dir", "results_cahn_hilliard")
#   Folder where model.pt, metrics.npz, plots, summary.json are saved.

# --- Global reproducibility and device --------------------------------------
SEED       = cfg.get("seed", 2)              # Master random seed
dtype_str  = cfg.get("dtype", "float64")     # "float64" for double-precision

# ---- Apply settings --------------------------------------------------------
# Set the default tensor dtype for ALL subsequently created tensors.
torch_dtype = torch.float64 if dtype_str == "float64" else torch.float32
torch.set_default_dtype(torch_dtype)

# Seed both PyTorch and NumPy for reproducibility.
torch.manual_seed(SEED)
np.random.seed(SEED)

# Use GPU if available, otherwise CPU.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Create the results directory (no error if it already exists).
os.makedirs(RESULTS_DIR, exist_ok=True)

# ---- Print a human-readable summary of what we just loaded -----------------
print(f"Config : {cfg['experiment']['name']}")
print(f"  PDE  : dU/dt = {D}*lap(lap U + {a2}*U + {a4}*U^3)")
print(f"  Domain: x=[{x_min},{x_max}], y=[{y_min},{y_max}], t=[{t_min},{t_max}]")
print(f"  Net  : {layer_dims}  activation={activation_name}")
if Nbfgs > 0:
    # Two-phase training: Adam warmup → quasi-Newton refinement
    print(f"  Train: Adam({Nepochs_ADAM}) -> {bfgs_variant if bfgs_method == 'BFGS' else bfgs_method}({Nbfgs})")
else:
    # Adam-only mode (no second phase)
    print(f"  Train: Adam-only({Nepochs_ADAM})")
print(f"  Samp : Nint={Nint}, N0={N0}, Nb={Nb}, RAD={'on' if rad_on else 'off'}")

In [ ]:
# ============================================================================
# CELL 3: NEURAL NETWORK DEFINITION
# ============================================================================
# Builds the MLP that will approximate U(t, x, y).
#
# Pipeline (for the canonical config):
#
#   (t, x, y) ──► InputNorm ──► Lin(3→20) ──► tanh ──► Lin(20→20) ──► tanh
#                 ──► Lin(20→20) ──► tanh ──► Lin(20→20) ──► tanh
#                 ──► Lin(20→1) ──► OutputScaling ──► Û(t,x,y)
#
# Key design decisions:
#   • InputNormalization maps each physical coordinate from [lo, hi] to
#     [-1, 1].  This keeps all activations in the sensitive (steep) region
#     of tanh and avoids vanishing / saturating gradients.
#   • The LAST linear layer uses variance-scaling initialisation
#     (TensorFlow-style).  This controls the initial output amplitude so
#     the loss starts at a reasonable value.
#   • OutputScaling multiplies the final output by a constant.  Useful if
#     the solution lives on a scale very different from O(1).
# ============================================================================

# ---- Activation function lookup -------------------------------------------
# Maps the string name from the YAML config to a PyTorch module class.
ACTIVATIONS = {
    "tanh":    nn.Tanh,      # smooth, bounded [-1,1] — default for PINNs
    "relu":    nn.ReLU,      # piecewise linear — fast but C⁰ only
    "gelu":    nn.GELU,      # smooth approximation of ReLU
    "silu":    nn.SiLU,      # x·σ(x), also called "swish"
    "sigmoid": nn.Sigmoid,   # bounded [0,1]
}


# ---------------------------------------------------------------------------
# InputNormalization: non-trainable affine layer
# ---------------------------------------------------------------------------
class InputNormalization(nn.Module):
    """
    Maps each input feature from [lo_i, hi_i] to [-1, 1].

    Formula per feature i:
        z_i = (x_i - mid_i) * (2 / (hi_i - lo_i))
    where  mid_i = (lo_i + hi_i) / 2

    Parameters are stored as *buffers* (not Parameters) so they are
    saved/loaded with the model but are NOT updated by the optimizer.
    """
    def __init__(self, lo, hi):
        super().__init__()
        # Convert Python lists to tensors with the global default dtype
        lo = torch.tensor(lo, dtype=torch.get_default_dtype())  # shape (3,)
        hi = torch.tensor(hi, dtype=torch.get_default_dtype())  # shape (3,)
        # register_buffer → saved in state_dict, moved with .to(device),
        #                    but NOT returned by .parameters().
        self.register_buffer("shift", 0.5 * (lo + hi))     # midpoint of each range
        self.register_buffer("scale", 2.0 / (hi - lo))     # maps half-range → 1

    def forward(self, x):
        # Subtract midpoint, then scale so the full range → [-1, 1]
        return (x - self.shift) * self.scale


# ---------------------------------------------------------------------------
# OutputScaling: multiply network output by a constant
# ---------------------------------------------------------------------------
class OutputScaling(nn.Module):
    """
    Simple pass-through that multiplies its input by a fixed scalar `s`.
    Placed after the final Linear layer when output_scaling ≠ 1.
    """
    def __init__(self, s):
        super().__init__()
        self.s = s           # stored as a plain Python float (not a Parameter)

    def forward(self, x):
        return x * self.s    # element-wise scaling of the (N, 1) output


# ---------------------------------------------------------------------------
# Variance-scaling initialisation (TF-style)
# ---------------------------------------------------------------------------
def variance_scaling_init(linear, scale=1.0):
    """
    Initialise a nn.Linear layer using TensorFlow's VarianceScaling rule:
        mode='fan_avg', distribution='uniform'

    The weight values are drawn from Uniform(-limit, limit) where
        limit = sqrt( 3 · scale / ( (fan_in + fan_out) / 2 ) )

    This keeps the variance of activations roughly constant across layers,
    which is important for training stability in deep networks.

    The bias is initialised to zero.
    """
    fi, fo = linear.in_features, linear.out_features   # fan_in, fan_out
    # Factor: 3·scale / (average of fan_in and fan_out)
    limit = math.sqrt(3.0 * scale / (0.5 * (fi + fo)))
    with torch.no_grad():                              # don't track this in autograd
        linear.weight.uniform_(-limit, limit)          # fill weights uniformly
        if linear.bias is not None:
            linear.bias.zero_()                        # bias starts at 0


# ---------------------------------------------------------------------------
# Main network class
# ---------------------------------------------------------------------------
class CahnHilliardNet(nn.Module):
    """
    Configurable MLP for the Cahn-Hilliard PINN.

    Input  : (t, x, y) — 3 features
    Output : U(t, x, y) — 1 feature

    Architecture (canonical):
      InputNorm → Linear(3→20) → tanh → [Linear(20→20) → tanh] × 3
                → Linear(20→1) → OutputScaling

    The constructor builds a nn.Sequential from the layer_dims list:
        layer_dims = [3, 20, 20, 20, 20, 1]
                      ↑                  ↑
                   input dim         output dim
    Every consecutive pair (d_i, d_{i+1}) becomes a Linear layer,
    followed by an activation — except the last pair (no activation
    after the output layer, because U can be any real number).
    """
    def __init__(self, layer_dims, activation_name="tanh",
                 output_scale=1.0, normalize=False, input_bounds=None):
        super().__init__()
        layers = []   # will be filled and wrapped in nn.Sequential

        # --- Optional input normalisation -----------------------------------
        # Placed BEFORE the first Linear layer.
        if normalize and input_bounds is not None:
            lo, hi = input_bounds   # e.g. [0, 0, 0], [20, 1, 1]
            layers.append(InputNormalization(lo, hi))

        # --- Hidden + output linear layers ----------------------------------
        # Look up the activation class from the string name.
        act_cls = ACTIVATIONS.get(activation_name, nn.Tanh)

        for i in range(len(layer_dims) - 1):
            # Add a Linear layer:  layer_dims[i] → layer_dims[i+1]
            layers.append(nn.Linear(layer_dims[i], layer_dims[i + 1]))

            # Add activation after every hidden layer, but NOT after the
            # output layer (last pair in the list).
            if i < len(layer_dims) - 2:
                layers.append(act_cls())

        # --- Optional output scaling ----------------------------------------
        if output_scale != 1.0:
            layers.append(OutputScaling(output_scale))

        # Wrap all layers into a single Sequential container.
        # Calling self.net(x) runs them in order.
        self.net = nn.Sequential(*layers)

        # --- Custom init for the LAST linear layer --------------------------
        # Walk modules in reverse to find the last nn.Linear.
        # Scale its weights accounting for output_scale so that the
        # initial network outputs have reasonable magnitude.
        for m in reversed(list(self.net.modules())):
            if isinstance(m, nn.Linear):
                # If output_scale = s, the effective weight is s·W, so
                # we reduce the init variance by 1/s² to compensate.
                sc = 1.0 / (output_scale ** 2) if output_scale != 0 else 1.0
                variance_scaling_init(m, scale=sc)
                break   # only initialise the last Linear layer

    def forward(self, x):
        """
        Forward pass.
        x : tensor of shape (N, 3) — columns are (t, x, y)
        Returns : tensor of shape (N, 1) — predicted U at each point.
                  (Actually (N, layer_dims[-1]), but that's 1.)
        """
        return self.net(x)


# ---- Build the network and move it to DEVICE ------------------------------

# Physical bounds of each input feature — used by InputNormalization.
input_bounds = ([t_min, x_min, y_min],    # lower bounds
                [t_max, x_max, y_max])     # upper bounds

# Instantiate the network with all settings from the config.
net = CahnHilliardNet(
    layer_dims,                       # e.g. (3, 20, 20, 20, 20, 1)
    activation_name=activation_name,  # e.g. "tanh"
    output_scale=output_scaling,      # e.g. 1.0
    normalize=normalize_inputs,       # True → prepend InputNormalization
    input_bounds=input_bounds,
).to(DEVICE)  # .to(DEVICE) moves all parameters to GPU if available

# ---- Report network size ---------------------------------------------------
# Count only *trainable* parameters (not buffers like InputNorm's shift/scale).
n_params = sum(p.numel() for p in net.parameters() if p.requires_grad)

# For dense BFGS the algorithm stores an n×n inverse-Hessian approximation.
# Each entry is a float64 = 8 bytes.  This is the dominant memory cost.
hess_gb = n_params ** 2 * 8 / 1e9   # approximate size in gigabytes

print(f"Trainable parameters : {n_params}")
print(f"Hessian approx memory: {n_params}x{n_params} = {n_params**2:,} entries  ({hess_gb:.2f} GB)")

In [ ]:
# ============================================================================
# CELL 4: INITIAL CONDITION AND SAMPLING FUNCTIONS
# ============================================================================
# This cell defines:
#   1. build_ic_interpolator — generates the random IC field on a grid and
#      wraps it in a smooth interpolator so we can query U₀(x, y) at
#      arbitrary (non-grid) points.
#   2. eval_ic — evaluates the IC interpolator on a batch of (t, x, y)
#      tensors, returning a PyTorch tensor on DEVICE.
#   3. sample_interior — draws random (t, x, y) inside the space-time domain
#      for the PDE residual loss.
#   4. sample_initial — draws random (t_min, x, y) for the IC loss.
#   5. sample_boundary_periodic — draws matched pairs on opposite faces
#      of the domain for the periodic-BC loss.
#
# All samplers return tensors on DEVICE (CPU or GPU) so downstream code
# (forward pass, autograd, loss) never needs a manual .to(DEVICE) call.
# ============================================================================

# ---------------------------------------------------------------------------
# 1. Build the IC interpolator
# ---------------------------------------------------------------------------
def build_ic_interpolator(ic_cfg, x_min, x_max, y_min, y_max):
    """
    Generate a smooth random initial condition U₀(x, y) and return
    a SciPy interpolator that can evaluate it at arbitrary coordinates.

    Steps:
      a) Draw a uniform random field on an (ny × nx) grid.
      b) Apply Gaussian smoothing (σ in grid cells, mode='wrap' for
         periodicity) so the field is smooth enough for a tanh net to
         represent.
      c) Pad the grid by one cell on each side with periodic wrapping
         so that cubic interpolation near the domain edges wraps correctly.
      d) Build a RegularGridInterpolator (cubic) over the padded grid.

    Returns:
        interp : RegularGridInterpolator  — callable(y, x) → U₀
        field  : np.ndarray (ny, nx)      — the raw (smoothed) field
    """
    # Create a NumPy RandomState with a fixed seed for reproducibility.
    rng   = np.random.RandomState(ic_cfg.get("seed", 42))
    nx    = ic_cfg.get("grid_nx", 128)             # Grid points in x
    ny    = ic_cfg.get("grid_ny", 128)             # Grid points in y
    sigma = ic_cfg.get("smoothing_sigma", 2.0)     # Gaussian kernel width
    low   = ic_cfg.get("low", -1.0)                # Uniform lower bound
    high  = ic_cfg.get("high", 1.0)                # Uniform upper bound

    # (a) Raw random field: each cell drawn independently from U[low, high].
    field = rng.uniform(low, high, size=(ny, nx))   # shape (128, 128)

    # (b) Gaussian smooth.  mode='wrap' treats the array as periodic, so the
    #     left/right and top/bottom edges blend into each other seamlessly.
    if sigma > 0:
        field = gaussian_filter(field, sigma=sigma, mode="wrap")

    # Grid coordinates — cell centres (not edges).
    dx = (x_max - x_min) / nx                       # cell width in x
    dy = (y_max - y_min) / ny                       # cell width in y
    xs = np.linspace(x_min + 0.5 * dx, x_max - 0.5 * dx, nx)  # (nx,)
    ys = np.linspace(y_min + 0.5 * dy, y_max - 0.5 * dy, ny)  # (ny,)

    # (c) Pad for periodic interpolation: add one ghost point at each end.
    #     np.pad mode='wrap' copies the opposite-edge value.
    xs_ext = np.concatenate([[x_min], xs, [x_max]])       # (nx + 2,)
    ys_ext = np.concatenate([[y_min], ys, [y_max]])       # (ny + 2,)
    field_ext = np.pad(field, ((1, 1), (1, 1)), mode="wrap")  # (ny+2, nx+2)

    # (d) Build a 2-D cubic interpolator.  Note: SciPy expects axes in the
    #     order matching the array dimensions → (y, x), not (x, y).
    interp = RegularGridInterpolator(
        (ys_ext, xs_ext),       # monotonic coordinate arrays
        field_ext,              # function values on the grid
        method="cubic",         # C² smooth interpolation
        bounds_error=False,     # don't raise if queried outside [y_min, y_max]
        fill_value=None,        # extrapolate instead of returning NaN
    )
    return interp, field


# Build the interpolator once at cell-execution time.
# ic_interp is used by eval_ic(); ic_field is kept for diagnostics / plotting.
ic_interp, ic_field = build_ic_interpolator(ic_cfg, x_min, x_max, y_min, y_max)


# ---------------------------------------------------------------------------
# 2. Evaluate the IC at arbitrary points
# ---------------------------------------------------------------------------
def eval_ic(X_t0):
    """
    Evaluate the initial condition at (x, y) locations.

    Args:
        X_t0 : tensor of shape (N, 3) with columns (t, x, y).
               The t column is ignored (IC only depends on space).

    Returns:
        tensor of shape (N, 1) on DEVICE with the IC values U₀(x, y).

    Internally:
        1. Extract x and y columns, move to CPU/numpy (interpolator is numpy).
        2. Query the RegularGridInterpolator with (y, x) ordering.
        3. Convert back to a PyTorch tensor on DEVICE.
    """
    x_np = X_t0[:, 1].detach().cpu().numpy()   # x coordinates as numpy array
    y_np = X_t0[:, 2].detach().cpu().numpy()   # y coordinates as numpy array

    # RegularGridInterpolator expects columns in axis order → (y, x).
    vals = ic_interp(np.column_stack([y_np, x_np]))   # shape (N,)

    # Convert back to a PyTorch tensor on the correct device and dtype.
    return torch.tensor(vals, dtype=torch.get_default_dtype(),
                        device=DEVICE).reshape(-1, 1)  # shape (N, 1)


# ---------------------------------------------------------------------------
# 3–5. Sampling functions
# ---------------------------------------------------------------------------
# Each sampler:
#   • Draws coordinates using NumPy RNG (fast, vectorised).
#   • Stacks them into a (N, 3) array with columns (t, x, y).
#   • Converts to a PyTorch tensor directly on DEVICE.

def sample_interior(n):
    """
    Sample n random interior collocation points (t, x, y).

    These are the points where we evaluate the PDE residual.
    Drawn uniformly over the full space-time domain:
        t ∈ [t_min, t_max],  x ∈ [x_min, x_max],  y ∈ [y_min, y_max]
    """
    t = t_min + (t_max - t_min) * np.random.rand(n)  # uniform in [t_min, t_max]
    x = x_min + Lx * np.random.rand(n)                # uniform in [x_min, x_max]
    y = y_min + Ly * np.random.rand(n)                # uniform in [y_min, y_max]
    # Stack into (N, 3) and convert to tensor on DEVICE.
    return torch.tensor(np.column_stack([t, x, y]),
                        dtype=torch.get_default_dtype(), device=DEVICE)


def sample_initial(n):
    """
    Sample n random initial-condition points (t_min, x, y).

    All points have t = t_min; x and y are uniform random.
    Used to compute L_IC = MSE( net(t_min, x, y) − U₀(x, y) ).
    """
    t = np.full(n, t_min)                        # all entries = t_min
    x = x_min + Lx * np.random.rand(n)           # random x
    y = y_min + Ly * np.random.rand(n)           # random y
    return torch.tensor(np.column_stack([t, x, y]),
                        dtype=torch.get_default_dtype(), device=DEVICE)


def sample_boundary_periodic(n):
    """
    Sample paired boundary points for periodic-BC enforcement.

    For a periodic domain the solution must satisfy:
        U(t, x_min, y) = U(t, x_max, y)     (x-periodicity)
        U(t, x, y_min) = U(t, x, y_max)     (y-periodicity)
    and optionally the normal derivatives must also match.

    We generate n // 2 random (t, y) pairs (or (t, x) pairs) and
    evaluate the network at both ends of the domain.  Each pair
    contributes one term to the MSE loss.

    Returns:
        X_xlo : (n//2, 3) points on the LEFT   face  (x = x_min)
        X_xhi : (n//2, 3) points on the RIGHT  face  (x = x_max)
        X_ylo : (n//2, 3) points on the BOTTOM face  (y = y_min)
        X_yhi : (n//2, 3) points on the TOP    face  (y = y_max)
    All tensors are on DEVICE.
    """
    n2 = max(n // 2, 1)  # half the budget for each direction (x and y)

    # --- x-periodic pairs (left ↔ right) ------------------------------------
    t_x = t_min + (t_max - t_min) * np.random.rand(n2)   # random times
    y_x = y_min + Ly * np.random.rand(n2)                 # random y coords
    # Left face: x = x_min
    X_xlo = torch.tensor(np.column_stack([t_x, np.full(n2, x_min), y_x]),
                          dtype=torch.get_default_dtype(), device=DEVICE)
    # Right face: x = x_max  (same t and y as the left face)
    X_xhi = torch.tensor(np.column_stack([t_x, np.full(n2, x_max), y_x]),
                          dtype=torch.get_default_dtype(), device=DEVICE)

    # --- y-periodic pairs (bottom ↔ top) ------------------------------------
    t_y = t_min + (t_max - t_min) * np.random.rand(n2)   # random times
    x_y = x_min + Lx * np.random.rand(n2)                 # random x coords
    # Bottom face: y = y_min
    X_ylo = torch.tensor(np.column_stack([t_y, x_y, np.full(n2, y_min)]),
                          dtype=torch.get_default_dtype(), device=DEVICE)
    # Top face: y = y_max  (same t and x as the bottom face)
    X_yhi = torch.tensor(np.column_stack([t_y, x_y, np.full(n2, y_max)]),
                          dtype=torch.get_default_dtype(), device=DEVICE)

    return X_xlo, X_xhi, X_ylo, X_yhi


# ---- Quick diagnostic: print IC stats to verify the interpolator -----------
print(f"IC field range: [{ic_field.min():.3f}, {ic_field.max():.3f}]  "
      f"shape: {ic_field.shape}  smoothing sigma: {ic_smooth_sig}")

In [ ]:
# ============================================================================
# CELL 5: PDE RESIDUAL — 4TH-ORDER AUTOGRAD COMPUTATION
# ============================================================================
#
# Goal: compute how badly the network violates the Cahn-Hilliard equation
# at a batch of interior collocation points.  If the residual is 0
# everywhere, the network is an exact solution.
#
# ---  The PDE  --------------------------------------------------------------
#
#   ∂U/∂t = D · ∇²( ∇²U + a₂·U + a₄·U³ )
#
# Expanding ∇² = ∂²/∂x² + ∂²/∂y² in 2-D:
#
#   ∂U/∂t = D · [ ∇⁴U  +  a₂·∇²U  +  a₄·∇²(U³) ]
#
# where the three spatial pieces are:
#
#   ∇⁴U    = U_xxxx + 2·U_xxyy + U_yyyy           (biharmonic)
#   ∇²U    = U_xx + U_yy                           (Laplacian of U)
#   ∇²(U³) = 6U·(U_x² + U_y²) + 3U²·(U_xx + U_yy)
#             ↑ this is the ANALYTICAL chain-rule expansion
#             Using it avoids computing ∂²(U³)/∂x² via 3 extra autograd
#             calls (which would be more expensive and numerically noisier).
#
# --- Derivative tree (8 autograd.grad calls) --------------------------------
#
#   Call 1: ∂U/∂X       → U_t, U_x, U_y          (1st order)
#   Call 2: ∂U_x/∂X     → U_xx                    (2nd order, x-direction)
#   Call 3: ∂U_y/∂X     → U_yy                    (2nd order, y-direction)
#   Call 4: ∂U_xx/∂X    → U_xxx, U_xxy            (3rd order)
#   Call 5: ∂U_yy/∂X    → U_yyy                   (3rd order)
#   Call 6: ∂U_xxx/∂X   → U_xxxx                  (4th order)
#   Call 7: ∂U_xxy/∂X   → U_xxyy                  (4th order, mixed)
#   Call 8: ∂U_yyy/∂X   → U_yyyy                  (4th order)
#
# Each call uses create_graph=True so that higher-order derivatives can
# be computed through the already-differentiated graph.
# ============================================================================


def forward_pass(net, X):
    """
    Run the network on input X and return U of shape (N, 1).

    net(X) returns shape (N, layer_dims[-1]).  We slice [:, 0:1] to keep
    it as (N, 1) — preserves the 2-D shape, which is convenient for
    broadcasting with autograd outputs.
    """
    return net(X)[:, 0:1]


def compute_pde_residual(net, X):
    """
    Compute the Cahn-Hilliard PDE residual at a batch of interior points.

    Args:
        net : CahnHilliardNet — the neural network
        X   : tensor (N, 3) — collocation points (t, x, y)

    Returns:
        U        : (N, 1) — network prediction at each point
        residual : (N,)   — PDE residual  r = U_t − D·[ ∇⁴U + a₂∇²U + a₄∇²(U³) ]
                             Should be ≈ 0 if the network satisfies the PDE.

    Implementation notes:
        • X is cloned and detached, then re-attached to the autograd graph
          via requires_grad_(True).  This ensures gradients flow through X
          (needed for spatial derivatives) without contaminating the
          upstream graph of whatever created X.
        • `ones` and `ones1` are the "grad_outputs" vectors.  In scalar-
          output autograd.grad, grad_outputs is the "seed" = ∂L/∂output.
          Since we want the raw Jacobian column dU/dX, we pass ones.
    """
    # --- Prepare X for differentiation --------------------------------------
    X = X.clone().detach().requires_grad_(True)
    # clone()   → new tensor, so we don't mutate the caller's X
    # detach()  → cuts any existing autograd history
    # requires_grad_(True) → enables grad computation w.r.t. X

    # --- Forward pass -------------------------------------------------------
    U = forward_pass(net, X)                  # shape (N, 1)

    # grad_outputs vectors: ones for 2-D output, ones1 for 1-D (flat) output
    ones  = torch.ones_like(U)                # (N, 1) — for U which is 2-D
    ones1 = torch.ones_like(U[:, 0])          # (N,)   — for flat scalar grads

    # ========================================================================
    # CALL 1: First derivatives of U w.r.t. (t, x, y)
    # ========================================================================
    # dU/dX has shape (N, 3) → columns are U_t, U_x, U_y
    dU  = torch.autograd.grad(U, X, grad_outputs=ones,
                              create_graph=True,    # keep graph for higher derivs
                              retain_graph=True)[0] # don't free the forward graph
    U_t = dU[:, 0]   # ∂U/∂t — the left-hand side of the PDE
    U_x = dU[:, 1]   # ∂U/∂x — needed for ∇²(U³) analytical formula
    U_y = dU[:, 2]   # ∂U/∂y — needed for ∇²(U³) analytical formula

    # ========================================================================
    # CALL 2: Second derivatives via x  →  U_xx
    # ========================================================================
    # Differentiate the scalar field U_x w.r.t. X → get (∂U_x/∂t, ∂U_x/∂x, ∂U_x/∂y)
    dUx = torch.autograd.grad(U_x, X, grad_outputs=ones1,
                              create_graph=True, retain_graph=True)[0]
    U_xx = dUx[:, 1]   # ∂²U/∂x² — component of the Laplacian

    # ========================================================================
    # CALL 3: Second derivatives via y  →  U_yy
    # ========================================================================
    dUy = torch.autograd.grad(U_y, X, grad_outputs=ones1,
                              create_graph=True, retain_graph=True)[0]
    U_yy = dUy[:, 2]   # ∂²U/∂y² — the other Laplacian component

    # ========================================================================
    # CALL 4: Third derivatives via xx  →  U_xxx, U_xxy
    # ========================================================================
    # Differentiate U_xx w.r.t. X.
    dUxx = torch.autograd.grad(U_xx, X, grad_outputs=ones1,
                               create_graph=True, retain_graph=True)[0]
    U_xxx = dUxx[:, 1]   # ∂³U/∂x³
    U_xxy = dUxx[:, 2]   # ∂³U/∂x²∂y  (mixed third derivative)

    # ========================================================================
    # CALL 5: Third derivatives via yy  →  U_yyy
    # ========================================================================
    dUyy = torch.autograd.grad(U_yy, X, grad_outputs=ones1,
                               create_graph=True, retain_graph=True)[0]
    U_yyy = dUyy[:, 2]   # ∂³U/∂y³

    # ========================================================================
    # CALL 6: Fourth derivative  →  U_xxxx
    # ========================================================================
    dUxxx = torch.autograd.grad(U_xxx, X, grad_outputs=ones1,
                                create_graph=True, retain_graph=True)[0]
    U_xxxx = dUxxx[:, 1]  # ∂⁴U/∂x⁴

    # ========================================================================
    # CALL 7: Fourth derivative  →  U_xxyy
    # ========================================================================
    dUxxy = torch.autograd.grad(U_xxy, X, grad_outputs=ones1,
                                create_graph=True, retain_graph=True)[0]
    U_xxyy = dUxxy[:, 2]  # ∂⁴U/∂x²∂y²  (mixed biharmonic term)

    # ========================================================================
    # CALL 8: Fourth derivative  →  U_yyyy
    # ========================================================================
    dUyyy = torch.autograd.grad(U_yyy, X, grad_outputs=ones1,
                                create_graph=True, retain_graph=True)[0]
    U_yyyy = dUyyy[:, 2]  # ∂⁴U/∂y⁴

    # ========================================================================
    # Assemble the PDE terms
    # ========================================================================

    # Biharmonic: ∇⁴U = U_xxxx + 2·U_xxyy + U_yyyy
    biharmonic = U_xxxx + 2.0 * U_xxyy + U_yyyy

    # Laplacian: ∇²U = U_xx + U_yy
    laplacian  = U_xx + U_yy

    # ∇²(U³) via the analytical chain rule (avoids 3 extra autograd calls):
    #
    #   ∂²(U³)/∂x² = 6U·U_x² + 3U²·U_xx
    #   ∂²(U³)/∂y² = 6U·U_y² + 3U²·U_yy
    #
    # Sum gives: ∇²(U³) = 6U·(U_x² + U_y²) + 3U²·(U_xx + U_yy)
    #                    = 6U·(U_x² + U_y²) + 3U²·laplacian
    Uf = U[:, 0]   # flatten from (N, 1) to (N,) for element-wise ops
    lap_U3 = 6.0 * Uf * (U_x ** 2 + U_y ** 2) + 3.0 * Uf ** 2 * laplacian

    # Full right-hand side:  D · [ ∇⁴U + a₂·∇²U + a₄·∇²(U³) ]
    rhs = D * (biharmonic + a2 * laplacian + a4 * lap_U3)

    # Residual = LHS − RHS.  Should be 0 if the PDE is satisfied.
    residual = U_t - rhs   # shape (N,)

    return U, residual


# ---------------------------------------------------------------------------
# Boundary derivative helper (for periodic BC with derivative matching)
# ---------------------------------------------------------------------------
def compute_bc_derivatives(net, X):
    """
    Compute U and its first spatial derivatives (U_x, U_y) at boundary points.

    Used when bc_deriv_order >= 1 to enforce that not only the values
    but also the normal derivatives match across periodic boundaries:
        U_x(x_min, y) = U_x(x_max, y)
        U_y(x, y_min) = U_y(x, y_max)

    Args:
        net : the neural network
        X   : (N, 3) boundary points

    Returns:
        U   : (N, 1) — network output
        U_x : (N,)   — ∂U/∂x
        U_y : (N,)   — ∂U/∂y
    """
    # Same clone-detach-require_grad pattern as compute_pde_residual.
    X = X.clone().detach().requires_grad_(True)
    U = forward_pass(net, X)                     # (N, 1)
    ones = torch.ones_like(U)                    # grad_outputs seed

    # Single autograd call: get dU/d(t, x, y) → extract x and y columns.
    dU = torch.autograd.grad(U, X, grad_outputs=ones,
                             create_graph=True,    # needed for backprop through loss
                             retain_graph=True)[0]
    # dU[:, 0] is U_t (not needed here), dU[:, 1] is U_x, dU[:, 2] is U_y.
    return U, dU[:, 1], dU[:, 2]

In [ ]:
# ============================================================================
# CELL 6: LOSS FUNCTION, GRADIENT COMPUTATION, AND TRAINING UTILITIES
# ============================================================================
#
# The total loss that the optimizers minimise is a weighted sum of three
# mean-squared-error (MSE) terms:
#
#   L_total = λ_pde · L_PDE  +  λ_ic · L_IC  +  λ_bc · L_BC
#
#   L_PDE : MSE of the PDE residual at interior collocation points.
#           → Forces the network to satisfy the Cahn-Hilliard equation.
#
#   L_IC  : MSE of ( network prediction − true IC ) at t = t_min.
#           → Forces the network to match U₀(x, y) at the initial time.
#
#   L_BC  : MSE of ( U_left − U_right ) + ( U_bottom − U_top )
#           → Forces periodic boundary conditions (values match).
#           Optionally also matches first derivatives if bc_deriv_order ≥ 1.
#
# This cell also defines:
#   • adam_step()           — one training iteration with Adam
#   • scipy_loss_and_grad() — flat numpy interface for SciPy's minimize()
#   • adaptive_rad()        — Residual-Adaptive Distribution resampling
# ============================================================================

# nn.MSELoss computes  (1/N) Σ (pred − target)²  over all elements.
mse = nn.MSELoss()


# ---------------------------------------------------------------------------
# Main loss function
# ---------------------------------------------------------------------------
def compute_loss(net, X_int, X_ic, bc_data):
    """
    Compute the total PINN loss for Cahn-Hilliard.

    Args:
        net     : the neural network
        X_int   : (N_int, 3)  interior collocation points for PDE residual
        X_ic    : (N_ic, 3)   initial-condition points (all at t = t_min)
        bc_data : tuple of 4 tensors (X_xlo, X_xhi, X_ylo, X_yhi)
                  paired boundary points for periodic matching

    Returns:
        total_loss : scalar tensor (differentiable — used for .backward())
        pde_loss   : plain float  (for logging only, detached from graph)
    """
    # Unpack the four boundary-point tensors.
    X_xlo, X_xhi, X_ylo, X_yhi = bc_data

    # ---- (1) PDE residual loss ---------------------------------------------
    # compute_pde_residual returns (U, residual) where residual should ≈ 0.
    _, residual = compute_pde_residual(net, X_int)
    # MSE of residual vs. zero → penalises PDE violations.
    zeros_r = torch.zeros_like(residual)     # target = 0 everywhere
    L_pde = mse(residual, zeros_r)           # scalar tensor

    # ---- (2) Initial-condition loss ----------------------------------------
    # Network prediction at t = t_min:
    U_ic_pred = forward_pass(net, X_ic)      # (N_ic, 1)
    # True IC from the interpolator:
    U_ic_true = eval_ic(X_ic)                # (N_ic, 1)
    # MSE between prediction and truth:
    L_ic = mse(U_ic_pred, U_ic_true)         # scalar tensor

    # ---- (3) Periodic boundary-condition loss ------------------------------
    if bc_deriv_order >= 1:
        # --- Value + first-derivative matching ------------------------------
        # For x-boundaries: compare U and U_x at x_min vs x_max.
        U_lo_x, Ux_lo, _ = compute_bc_derivatives(net, X_xlo)  # left face
        U_hi_x, Ux_hi, _ = compute_bc_derivatives(net, X_xhi)  # right face
        # For y-boundaries: compare U and U_y at y_min vs y_max.
        U_lo_y, _, Uy_lo  = compute_bc_derivatives(net, X_ylo)  # bottom face
        U_hi_y, _, Uy_hi  = compute_bc_derivatives(net, X_yhi)  # top face

        # Value matching:       U(left)   = U(right),  U(bottom) = U(top)
        # Derivative matching:  U_x(left) = U_x(right), U_y(bottom) = U_y(top)
        L_bc = (mse(U_lo_x, U_hi_x) + mse(U_lo_y, U_hi_y)
                + mse(Ux_lo, Ux_hi) + mse(Uy_lo, Uy_hi))
    else:
        # --- Value matching only (bc_deriv_order == 0) ----------------------
        # Just check that U matches; no derivative check.
        U_lo_x = forward_pass(net, X_xlo)   # U at left face
        U_hi_x = forward_pass(net, X_xhi)   # U at right face
        U_lo_y = forward_pass(net, X_ylo)   # U at bottom face
        U_hi_y = forward_pass(net, X_yhi)   # U at top face
        L_bc = mse(U_lo_x, U_hi_x) + mse(U_lo_y, U_hi_y)

    # ---- Weighted total loss -----------------------------------------------
    total = lam_pde * L_pde + lam_ic * L_ic + lam_bc * L_bc
    # Return the total (graph-attached) and pde loss (detached float).
    return total, float(L_pde.detach())


# ---------------------------------------------------------------------------
# Adam training step
# ---------------------------------------------------------------------------
def adam_step(net, optimizer, X_int, X_ic, bc_data):
    """
    Execute one Adam training step:
        1. Zero all parameter gradients.
        2. Forward pass → compute loss.
        3. Backward pass → compute gradients.
        4. optimizer.step() → update weights.

    Returns:
        loss_val : float — total loss (for logging)
        pde_val  : float — PDE component of the loss (for logging)
    """
    # Zero gradients manually (faster than optimizer.zero_grad() with set_to_none).
    for p in net.parameters():
        p.grad = None                      # sets grad to None instead of zeroing

    # Compute the loss and PDE sub-loss.
    loss_val, pde_val = compute_loss(net, X_int, X_ic, bc_data)

    # Backpropagate: compute ∂loss/∂θ for every parameter θ.
    loss_val.backward()

    # Adam update: θ ← θ − lr · m̂/(√v̂ + ε)  (simplified; Adam tracks
    # first and second moment estimates m̂, v̂ per parameter).
    optimizer.step()

    return float(loss_val.detach()), pde_val


# ---------------------------------------------------------------------------
# SciPy-compatible loss + gradient wrapper (for BFGS phase)
# ---------------------------------------------------------------------------
def scipy_loss_and_grad(weights, net, X_int, X_ic, bc_data, power=1.0):
    """
    Interface between SciPy's minimize (works with flat numpy arrays) and
    PyTorch's autograd (works with Parameter tensors).

    Steps:
        1. Copy the flat numpy weight vector into the network's Parameters.
        2. Compute the PINN loss via compute_loss().
        3. Optionally apply a power transform: minimise L^(1/p) instead of L.
           Why?  For p > 1 the landscape becomes flatter, which improves
           the condition number and helps quasi-Newton methods converge.
        4. Compute gradients dL/dθ via autograd.
        5. Return (loss_scalar, gradient_flat_numpy) — the format that
           scipy.optimize.minimize with jac=True expects.

    Args:
        weights : 1-D numpy array of length n_params — current parameter vector
        net     : the neural network (its Parameters will be overwritten)
        X_int   : interior collocation points (tensor on DEVICE)
        X_ic    : initial-condition points    (tensor on DEVICE)
        bc_data : tuple of 4 boundary tensors (on DEVICE)
        power   : float — power-transform exponent (1.0 = no transform)

    Returns:
        (float, 1-D numpy array) — loss value and gradient vector
    """
    # Identify device and dtype from the network's own parameters.
    device = next(net.parameters()).device
    dtype  = next(net.parameters()).dtype

    # Convert numpy weights to a PyTorch tensor on the same device.
    w = torch.as_tensor(weights, dtype=dtype, device=device)

    # Load the flat weight vector back into the network's parameter buffers.
    # This is an in-place operation — after this, net(X) uses the new weights.
    with torch.no_grad():
        vector_to_parameters(w, net.parameters())

    # Compute the loss (graph is built here).
    loss_val, _ = compute_loss(net, X_int, X_ic, bc_data)

    # Optional power transform: L_eff = L^(1/p).
    # For logging, we undo this in the callback: L = L_eff^p.
    loss_eff = loss_val if power == 1.0 else loss_val ** (1.0 / power)

    # Compute gradients of loss_eff w.r.t. each parameter tensor.
    params = list(net.parameters())
    grads = torch.autograd.grad(
        loss_eff, params,
        create_graph=False,    # no higher-order derivatives needed
        retain_graph=False,    # free the graph after this (saves memory)
        allow_unused=False,    # every param should have a gradient
    )

    # Flatten all per-layer gradient tensors into a single 1-D numpy array.
    g_flat = torch.cat([g.reshape(-1) for g in grads]).detach().cpu().numpy()

    # Return the (scalar, gradient) pair that SciPy expects.
    return float(loss_eff.detach().cpu()), g_flat


# ---------------------------------------------------------------------------
# RAD: Residual-Adaptive Distribution resampling
# ---------------------------------------------------------------------------
def adaptive_rad(net, n_int, rad_args, n_cand=50000):
    """
    Resample interior collocation points, focusing on high-residual regions.

    Algorithm:
        1. Draw n_cand candidate points uniformly in the domain.
        2. Evaluate |PDE residual| at each candidate (cheap: no backward).
        3. Compute sampling weights:  w_i = |r_i|^k1
        4. Normalised probability:    p_i = (w_i / mean(w) + k2) / sum(…)
           The offset k2 ensures even low-residual regions keep some points.
        5. Subsample n_int points WITHOUT replacement, weighted by p.

    Why this helps:  Default uniform sampling wastes points in regions
    where the PDE is already well-satisfied.  RAD concentrates points
    where the error is largest, accelerating convergence.

    Args:
        net     : the neural network
        n_int   : how many interior points to keep
        rad_args: (k1, k2) — exponent and offset for the weight formula
        n_cand  : how many candidate points to evaluate (default 50 000)

    Returns:
        tensor (n_int, 3) — selected collocation points on DEVICE
    """
    # Step 1: draw candidates uniformly.
    X_cand = sample_interior(n_cand)

    k1, k2 = rad_args

    # Step 2: evaluate |PDE residual| at each candidate (no grad needed
    # because we're just computing weights, not training).
    Y = torch.abs(compute_pde_residual(net, X_cand)[1]).detach().reshape(-1)
    #   [1] → the residual (index 0 is U, index 1 is residual)
    #   .detach() → we don't need gradients for this

    # Step 3–4: compute normalised sampling probabilities.
    w = Y ** k1                                # weight ∝ |residual|^k1
    p = (w / w.mean() + k2)                    # shift: ensures p > 0 everywhere
    p = (p / p.sum()).clamp_min(1e-12)          # normalise to a valid distribution

    # Step 5: weighted subsampling without replacement.
    ids = torch.multinomial(p, num_samples=n_int, replacement=False)

    # Return the selected collocation points.
    return X_cand[ids]

In [ ]:
# ============================================================================
# CELL 7: PHASE 1 — ADAM TRAINING (FIRST-ORDER WARMUP)
# ============================================================================
# Adam is a first-order optimizer: it only uses the gradient ∂L/∂θ, not
# any curvature (Hessian) information.  It's cheap per iteration and
# robust, but converges slowly near a minimum.
#
# We use Adam to move the network from its random initialisation into a
# "roughly good" region of parameter space.  Then the quasi-Newton
# optimizer (BFGS) takes over for fine-tuning with curvature info.
#
# If the YAML config sets adam.epochs = 0, this phase is skipped entirely
# (useful for testing BFGS-from-scratch configurations).
# ============================================================================

# ---- Generate the first batch of training data -----------------------------
# These tensors are used for the first Adam epoch.  They'll be replaced
# every `Nresample` epochs (see the if-block inside the loop).
X_int   = sample_interior(Nint)              # (Nint, 3) — interior (PDE residual)
X_ic    = sample_initial(N0)                 # (N0, 3)   — IC points at t = t_min
bc_data = sample_boundary_periodic(Nb)       # 4 tensors  — boundary pairs

# ---- Containers for tracking the loss curve --------------------------------
adam_losses = []         # will hold one float per epoch (total loss)
adam_t0 = perf_counter() # start the wall-clock timer

if Nepochs_ADAM > 0:
    # ---- Configure the Adam optimizer --------------------------------------
    # net.parameters() yields all trainable tensors (weights and biases).
    optimizer = torch.optim.Adam(
        net.parameters(),
        lr=adam_lr,              # initial learning rate (e.g. 1e-3)
        betas=adam_betas,        # (β₁, β₂) — momentum decay rates
        eps=adam_eps,            # ε added to denominator for numerical stability
    )

    # Exponential LR decay: lr(step) = adam_lr · decay_rate^(step / decay_steps)
    # This gradually reduces the learning rate over time for finer updates.
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        lr_lambda=lambda step: lr_decay_rate ** (step / lr_decay_steps)
    )

    # ---- Training loop -----------------------------------------------------
    for epoch in range(Nepochs_ADAM):

        # --- Periodic resampling of collocation points ----------------------
        # Every Nresample epochs, draw fresh points.  If RAD is enabled,
        # the new interior points are concentrated in high-residual regions.
        if (epoch + 1) % Nresample == 0:
            if rad_on:
                # RAD: sample proportional to |PDE residual|
                X_int = adaptive_rad(net, Nint, rad_args, Nsampling)
            else:
                # Uniform: just redraw randomly
                X_int = sample_interior(Nint)
            # Also refresh IC and BC points (uniform random).
            X_ic    = sample_initial(N0)
            bc_data = sample_boundary_periodic(Nb)

        # --- One Adam step --------------------------------------------------
        # Forward → loss → backward → parameter update.
        loss_v, pde_v = adam_step(net, optimizer, X_int, X_ic, bc_data)

        # Decay the learning rate according to the schedule.
        scheduler.step()

        # Record the total loss for this epoch.
        adam_losses.append(loss_v)

        # --- Periodic logging -----------------------------------------------
        if (epoch + 1) % Nprint == 0:
            print(f"Adam  epoch {epoch+1:5d}/{Nepochs_ADAM}  "
                  f"loss={loss_v:.4e}  pde={pde_v:.4e}")

# ---- Report Adam timing ---------------------------------------------------
adam_time = perf_counter() - adam_t0   # seconds elapsed during Adam phase
if adam_losses:
    print(f"\nAdam phase complete: {adam_time:.1f}s  "
          f"final loss={adam_losses[-1]:.4e}")
else:
    print("Adam phase skipped (0 epochs).")

In [ ]:
# ============================================================================
# CELL 8: PHASE 2 — QUASI-NEWTON TRAINING (BFGS / L-BFGS-B)
# ============================================================================
# After Adam has warmed up the network, we switch to a quasi-Newton
# optimizer that uses an approximation to the inverse Hessian H⁻¹.
# This gives the optimizer curvature information, allowing it to take
# much larger, well-directed steps — often reducing loss by orders of
# magnitude relative to Adam for the same number of function evaluations.
#
# Two families are supported (selected by bfgs_method in the YAML):
#
#   "BFGS"     — Full dense inverse-Hessian approximation (n×n matrix).
#                Variants (via bfgs_variant in the YAML):
#                  BFGS_scipy   — standard BFGS update
#                  SSBFGS_OL    — Self-Scaled BFGS (Oren–Luenberger)
#                  SSBFGS_AB    — Self-Scaled BFGS (Al-Baali)
#                  SSBroyden1   — Self-Scaled Broyden class 1
#                  SSBroyden2   — Self-Scaled Broyden class 2
#                These variants live in the PATCHED SciPy _optimize.py.
#                The inverse Hessian H_k is WARM-STARTED across batches:
#                after resampling, the new minimisation starts from where
#                the old one left off (both weights AND H_k).
#
#   "L-BFGS-B" — Limited-memory BFGS.  Instead of a dense n×n matrix,
#                stores only `maxcor` pairs of (s, y) vectors.  Much
#                less memory (O(n·maxcor) vs O(n²)), but no warm-start
#                across batches (SciPy doesn't expose the internal state).
#
# If Nbfgs == 0 (Adam-only experiments), this entire cell is a no-op.
#
# Key design: scipy.optimize.minimize is called in a LOOP.  Each call
# runs for at most Nchange iterations, then we resample collocation
# points (RAD) and call minimize again.  This prevents the optimizer
# from overfitting to a fixed set of collocation points.
# ============================================================================

if Nbfgs > 0:
    # ---- Setup: flatten current weights into a numpy vector ----------------
    # parameters_to_vector concatenates all parameter tensors into 1-D.
    # .detach() removes them from the autograd graph (we just want values).
    # .cpu().numpy() converts to a numpy array for SciPy.
    initial_weights = parameters_to_vector(
        [p.detach() for p in net.parameters()]
    ).cpu().numpy()

    # ---- Decide whether we use dense BFGS or limited-memory ----------------
    use_dense_hessian = (bfgs_method == "BFGS")

    if use_dense_hessian:
        # Initialise the inverse-Hessian approximation as the identity matrix.
        # This is the standard starting point: H₀ = I → first step is steepest descent.
        H0 = np.eye(initial_weights.size, dtype=np.float64)   # shape (n_params, n_params)

    # ---- Iteration counter (persists across batches) -----------------------
    cont = 0                               # total BFGS iterations completed so far

    # ---- Arrays for logging ------------------------------------------------
    n_ckpts = Nbfgs // Nprint              # how many checkpoints will be recorded
    bfgs_losses  = np.zeros(n_ckpts)       # loss at each checkpoint
    bfgs_pde     = np.zeros(n_ckpts)       # PDE loss at each checkpoint (unused currently)

    initial_scale = bfgs_init_scale        # whether to scale H₀ on the first batch
    power = bfgs_power                     # power-transform exponent (1 = no transform)

    # ---- Callback: invoked by SciPy after EVERY BFGS iteration -------------
    def bfgs_callback(*, intermediate_result):
        """
        SciPy calls this after each BFGS iteration.
        We use it to log the loss at regular intervals.

        intermediate_result is a SciPy OptimizeResult with .fun (loss value)
        and .x (current weight vector).
        """
        global cont
        cont += 1                          # increment global counter

        # Only log every Nprint iterations.
        if cont % Nprint != 0:
            return

        # Compute the checkpoint index.
        idx = cont // Nprint - 1
        if idx < 0 or idx >= n_ckpts:
            return                          # safety: index out of range

        # Read the loss value from the SciPy result.
        loss_val = float(intermediate_result.fun)

        # If we're using a power transform (L^(1/p)), undo it for logging
        # so we see the real loss, not the transformed one.
        if power != 1.0:
            loss_val = loss_val ** power    # L_eff^p = (L^(1/p))^p = L

        bfgs_losses[idx] = loss_val

        # Print a progress line.
        optimizer_label = bfgs_variant if use_dense_hessian else bfgs_method
        print(f"{optimizer_label}  iter {cont:5d}/{Nbfgs}  loss={loss_val:.4e}")

    # ---- Build the options dict for scipy.optimize.minimize ----------------
    def build_bfgs_options():
        """
        Construct the `options` dict that SciPy's minimize() accepts.

        For BFGS:
            maxiter      — max iterations this batch (= Nchange)
            gtol = 0     — never stop early due to gradient tolerance
            hess_inv0    — warm-started inverse Hessian (our H0 matrix)
            method_bfgs  — selects the self-scaling variant (patched SciPy)
            initial_scale — whether to scale H₀ by (y^T s)/(y^T y)

        For L-BFGS-B:
            maxiter      — max iterations this batch
            maxcor       — number of (s, y) correction pairs to store
            ftol = 0     — never stop early due to function tolerance
        """
        opts = {"maxiter": Nchange, "gtol": 0}

        if bfgs_method == "BFGS":
            opts["hess_inv0"]     = H0              # warm-started H⁻¹
            opts["method_bfgs"]   = bfgs_variant    # e.g. "SSBroyden2"
            opts["initial_scale"] = initial_scale   # scale H₀ on first batch?
        elif bfgs_method == "L-BFGS-B":
            opts["maxcor"]  = bfgs_maxcor           # memory budget (default 20)
            opts["ftol"]    = 0                     # don't stop on small Δf
        return opts

    # ---- Main BFGS loop: repeat until total iterations reach Nbfgs ---------
    bfgs_t0 = perf_counter()               # start wall-clock timer

    while cont < Nbfgs:
        # Call SciPy's minimize for one batch of Nchange iterations.
        result = minimize(
            scipy_loss_and_grad,            # objective function (returns loss, grad)
            initial_weights,                # starting point (flat numpy vector)
            args=(net, X_int, X_ic, bc_data, power),  # extra args passed to objective
            method=bfgs_method,             # "BFGS" or "L-BFGS-B"
            jac=True,                       # objective returns (loss, gradient) tuple
            options=build_bfgs_options(),   # method-specific settings
            tol=0,                          # never stop early on tolerance
            callback=bfgs_callback,         # called after each iteration
        )

        # ---- Extract results from this batch ------------------------------
        # Update the weight vector for the next batch.
        initial_weights = result.x          # flat numpy array of optimised weights

        if use_dense_hessian:
            # Retrieve the updated inverse-Hessian from SciPy's result.
            H0 = result.hess_inv

            # Symmetrise: H should be symmetric, but floating-point drift can
            # introduce tiny asymmetries.  Averaging H and H^T fixes this.
            H0 = 0.5 * (H0 + H0.T)

            # Positive-definiteness check.  The BFGS update theorem guarantees
            # PD if the Wolfe conditions hold, but numerical noise can break this.
            # If H is no longer PD, reset to identity (loses curvature info but
            # is safe — effectively restarts steepest descent).
            try:
                cholesky(H0)               # succeeds only if PD
            except LinAlgError:
                print("  [!] H lost positive-definiteness — resetting to I")
                H0 = np.eye(len(initial_weights), dtype=np.float64)

        # ---- Resample collocation points (RAD) for the next batch ----------
        # This is the reason for the loop: fresh points prevent overfitting.
        if rad_on:
            X_int = adaptive_rad(net, Nint, rad_args, Nsampling)
        else:
            X_int = sample_interior(Nint)
        X_ic    = sample_initial(N0)
        bc_data = sample_boundary_periodic(Nb)

        # Only scale H₀ on the very first batch (if enabled).
        # After that, H₀ is warm-started and should not be re-scaled.
        initial_scale = False

    # ---- Report BFGS timing ------------------------------------------------
    bfgs_time = perf_counter() - bfgs_t0
    optimizer_label = bfgs_variant if use_dense_hessian else bfgs_method
    print(f"\n{optimizer_label} phase complete: {bfgs_time:.1f}s")
    print(f"Total training time: {adam_time + bfgs_time:.1f}s")

else:
    # ---- Adam-only mode: no second-order phase -----------------------------
    # Set variables that Cell 9 (results) expects, even though BFGS didn't run.
    bfgs_time = 0.0
    n_ckpts = 0
    bfgs_losses = np.array([])
    bfgs_pde    = np.array([])
    print(f"\nAdam-only mode: no BFGS phase.  Total time: {adam_time:.1f}s")

In [ ]:
# ============================================================================
# CELL 9: SAVE RESULTS, VISUALIZATION, AND SUMMARY
# ============================================================================
# Saves:
#   1. Model weights            → {RESULTS_DIR}/model.pt
#   2. Loss history + timing    → {RESULTS_DIR}/metrics.npz
#   3. Config snapshot (YAML)   → {RESULTS_DIR}/config_used.yaml
#   4. Loss curve plot           → {RESULTS_DIR}/loss_curve.png
#   5. Solution snapshots        → {RESULTS_DIR}/snapshots.png
#   6. IC comparison plot        → {RESULTS_DIR}/ic_comparison.png
#   7. Mass conservation plot    → {RESULTS_DIR}/mass_conservation.png
# ============================================================================

import json

os.makedirs(RESULTS_DIR, exist_ok=True)
optimizer_label = (bfgs_variant if bfgs_method == "BFGS" else bfgs_method) if Nbfgs > 0 else "none"

# ---- 0. Save model, metrics, and config -----------------------------------

# Model weights
torch.save(net.state_dict(), os.path.join(RESULTS_DIR, "model.pt"))
print(f"Model saved to {RESULTS_DIR}/model.pt")

# Loss history + timing as compressed numpy archive
np.savez_compressed(
    os.path.join(RESULTS_DIR, "metrics.npz"),
    adam_losses=np.array(adam_losses),
    bfgs_losses=bfgs_losses if Nbfgs > 0 else np.array([]),
    adam_time=adam_time,
    bfgs_time=bfgs_time if Nbfgs > 0 else 0.0,
    total_time=adam_time + (bfgs_time if Nbfgs > 0 else 0.0),
    n_params=n_params,
)
print(f"Metrics saved to {RESULTS_DIR}/metrics.npz")

# Copy of the config used (for reproducibility)
with open(os.path.join(RESULTS_DIR, "config_used.yaml"), "w") as f:
    yaml.dump(cfg, f, default_flow_style=False)
print(f"Config saved to {RESULTS_DIR}/config_used.yaml")

# JSON summary for easy parsing
summary = {
    "experiment": cfg["experiment"]["name"],
    "optimizer": optimizer_label,
    "adam_epochs": Nepochs_ADAM,
    "bfgs_iters": Nbfgs,
    "adam_time_s": round(adam_time, 2),
    "bfgs_time_s": round(bfgs_time if Nbfgs > 0 else 0.0, 2),
    "total_time_s": round(adam_time + (bfgs_time if Nbfgs > 0 else 0.0), 2),
    "final_adam_loss": adam_losses[-1] if adam_losses else None,
    "final_bfgs_loss": float(bfgs_losses[bfgs_losses > 0][-1]) if Nbfgs > 0 and (bfgs_losses > 0).any() else None,
    "n_params": n_params,
}
with open(os.path.join(RESULTS_DIR, "summary.json"), "w") as f:
    json.dump(summary, f, indent=2)
print(f"Summary saved to {RESULTS_DIR}/summary.json")

# ---- 1. Loss curve --------------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 4))
ax.semilogy(range(1, len(adam_losses) + 1), adam_losses, label="Adam", alpha=0.8)
if Nbfgs > 0:
    bfgs_epochs = Nepochs_ADAM + np.arange(1, n_ckpts + 1) * Nprint
    valid = bfgs_losses > 0
    if valid.any():
        ax.semilogy(bfgs_epochs[valid], bfgs_losses[valid],
                     label=optimizer_label, alpha=0.8)
    ax.axvline(Nepochs_ADAM, color="gray", ls="--", lw=0.8, label="Adam→BFGS")
ax.set_xlabel("Iteration")
ax.set_ylabel("Total Loss")
ax.set_title(f"Cahn-Hilliard PINN — {cfg['experiment']['name']}")
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "loss_curve.png"), dpi=150)
plt.show()

# ---- 2. Solution snapshots ------------------------------------------------
n_snap = 5
snap_times = np.linspace(t_min, t_max, n_snap)
nx_plot, ny_plot = 64, 64
xs_plot = np.linspace(x_min, x_max, nx_plot)
ys_plot = np.linspace(y_min, y_max, ny_plot)
xx, yy = np.meshgrid(xs_plot, ys_plot)

fig, axes = plt.subplots(1, n_snap, figsize=(4 * n_snap, 3.5))
net.eval()
for i, t_val in enumerate(snap_times):
    tt = np.full_like(xx, t_val)
    X_plot = torch.tensor(
        np.column_stack([tt.ravel(), xx.ravel(), yy.ravel()]),
        dtype=torch.get_default_dtype(), device=DEVICE,
    )
    with torch.no_grad():
        U_plot = forward_pass(net, X_plot).cpu().numpy().reshape(ny_plot, nx_plot)
    im = axes[i].pcolormesh(xs_plot, ys_plot, U_plot, cmap="RdBu_r", shading="auto")
    axes[i].set_title(f"t = {t_val:.1f}")
    axes[i].set_aspect("equal")
    fig.colorbar(im, ax=axes[i], fraction=0.046)
net.train()
fig.suptitle("Predicted U(t, x, y)", fontsize=14, y=1.02)
fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "snapshots.png"), dpi=150, bbox_inches="tight")
plt.show()

# ---- 3. IC comparison -----------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
# True IC (interpolated)
X_ic_plot = torch.tensor(
    np.column_stack([np.full(nx_plot * ny_plot, t_min), xx.ravel(), yy.ravel()]),
    dtype=torch.get_default_dtype(), device=DEVICE,
)
U_ic_true_plot = eval_ic(X_ic_plot).cpu().numpy().reshape(ny_plot, nx_plot)
im0 = axes[0].pcolormesh(xs_plot, ys_plot, U_ic_true_plot, cmap="RdBu_r", shading="auto")
axes[0].set_title("True IC  U₀(x,y)")
axes[0].set_aspect("equal")
fig.colorbar(im0, ax=axes[0], fraction=0.046)

# Predicted IC
net.eval()
with torch.no_grad():
    U_ic_pred_plot = forward_pass(net, X_ic_plot).cpu().numpy().reshape(ny_plot, nx_plot)
net.train()
im1 = axes[1].pcolormesh(xs_plot, ys_plot, U_ic_pred_plot, cmap="RdBu_r", shading="auto")
axes[1].set_title("PINN at t = 0")
axes[1].set_aspect("equal")
fig.colorbar(im1, ax=axes[1], fraction=0.046)

fig.suptitle("Initial Condition Comparison", fontsize=14, y=1.02)
fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "ic_comparison.png"), dpi=150, bbox_inches="tight")
plt.show()

# ---- 4. Mass conservation diagnostic --------------------------------------
# The Cahn-Hilliard equation with periodic BCs conserves spatial average:
#     d/dt ∫_Ω U dx dy = 0
# Evaluate mean(U) on a fixed grid at several times. Any drift indicates
# the PINN is not faithfully reproducing the conservation law.
n_mass_times = 21
mass_times = np.linspace(t_min, t_max, n_mass_times)
mean_U = np.zeros(n_mass_times)

net.eval()
with torch.no_grad():
    for i, t_val in enumerate(mass_times):
        tt = np.full(nx_plot * ny_plot, t_val)
        X_mass = torch.tensor(
            np.column_stack([tt, xx.ravel(), yy.ravel()]),
            dtype=torch.get_default_dtype(), device=DEVICE,
        )
        U_mass = forward_pass(net, X_mass).cpu().numpy().ravel()
        mean_U[i] = U_mass.mean()
net.train()

mass_drift = mean_U.max() - mean_U.min()
print(f"\nMass conservation diagnostic (⟨U⟩ over Ω at {n_mass_times} times):")
print(f"  ⟨U⟩(t={t_min}) = {mean_U[0]:+.6f}")
print(f"  ⟨U⟩(t={t_max}) = {mean_U[-1]:+.6f}")
print(f"  max drift       = {mass_drift:.2e}")

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(mass_times, mean_U, "o-", markersize=4, linewidth=1.5)
ax.axhline(mean_U[0], color="gray", ls="--", lw=0.8, label=f"⟨U⟩(t₀) = {mean_U[0]:.4f}")
ax.set_xlabel("t")
ax.set_ylabel("⟨U⟩(t)")
ax.set_title(f"Mass Conservation — drift = {mass_drift:.2e}")
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "mass_conservation.png"), dpi=150)
plt.show()

# Add mass data to summary
summary["mass_mean_t0"] = round(float(mean_U[0]), 6)
summary["mass_mean_tmax"] = round(float(mean_U[-1]), 6)
summary["mass_drift"] = round(float(mass_drift), 8)
with open(os.path.join(RESULTS_DIR, "summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

# ---- 5. Summary -----------------------------------------------------------
print(f"\nExperiment: {cfg['experiment']['name']}")
print(f"  Adam time : {adam_time:.1f}s  ({Nepochs_ADAM} epochs)")
if Nbfgs > 0:
    print(f"  BFGS time : {bfgs_time:.1f}s  ({Nbfgs} iters, {optimizer_label})")
print(f"  Total time: {adam_time + bfgs_time:.1f}s")
print(f"  Final loss: {adam_losses[-1]:.4e}" if Nbfgs == 0 else
      f"  Final BFGS: {bfgs_losses[bfgs_losses > 0][-1]:.4e}" if (bfgs_losses > 0).any() else
      f"  Final loss: {adam_losses[-1]:.4e}")
print(f"Results saved to {RESULTS_DIR}/")